# 01: Data exploration

Loads the dataset and runs the Phase 1 profiler. Logic lives in `src/ai_experiment_lab/data/`; this notebook only calls it.

**Before running:** upload the dataset CSV to `MyDrive/ai-experiment-lab/data/` and set `DATASET_FILE` below.

In [ ]:
# --- AI Experiment Lab: standard Colab setup cell ---------------------------------
import os, subprocess, sys

REPO_URL = "https://github.com/sudhanshu230230/AI_Experiment_Lab.git"   # <-- edit
REPO_DIR = "/content/ai-experiment-lab"
DRIVE_ROOT = "/content/drive/MyDrive/ai-experiment-lab"
EXTRAS = "ml,tracking"          # pip extras from pyproject.toml

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/mlflow", exist_ok=True)

# For a private repo, store a token in Colab Secrets as GITHUB_TOKEN (never paste it here).
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

# An editable install adds a .pth file, which a running kernel does not re-read.
# Put src/ on sys.path so the import works in this session without a restart.
src_dir = f"{REPO_DIR}/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Tracking location: persistent Drive storage (see docs/decisions/ADR-001-mlflow-storage.md).
os.environ["MLFLOW_TRACKING_URI"] = f"file:{DRIVE_ROOT}/mlflow"
os.environ["AEL_DATA_DIR"] = f"{DRIVE_ROOT}/data"

commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Code version (commit):", commit)
print("Data dir:", os.environ["AEL_DATA_DIR"])

import ai_experiment_lab
print("ai_experiment_lab", ai_experiment_lab.__version__, "imported OK")

In [ ]:
from pathlib import Path
from ai_experiment_lab.data import load_csv, profile_dataframe, format_report
from ai_experiment_lab.utils.config import get_data_dir, load_config

DATASET_FILE = "telco_churn.csv"     # <-- file name in your Drive data folder
TARGET = "Churn"

df, info = load_csv(Path(get_data_dir()) / DATASET_FILE)
print(info)          # record sha256 in the phase report (dataset version)
df.head()

In [ ]:
profile = profile_dataframe(df, target=TARGET)
print(format_report(profile))

In [ ]:
# Raw dtypes and a closer look at anything the report flagged
display(df.dtypes.to_frame("dtype").T)
for col in profile["numeric_like_text_features"]:
    print(col, "-> non-numeric values:", df[col][df[col].astype(str).str.strip().eq("")].shape[0], "blank")

## Observations

Write findings here, then copy the conclusions into `docs/phases/phase-01-*.md`:

- Missing / blank values:
- Columns needing type conversion:
- Identifier columns to exclude:
- Class balance:
- Surprises: